# CNN/DailyMail Data Split

This notebook creates:

- 800 complete training pairs
- 100 complete validation pairs
- 100 test records

Training and validation use only records where the full prompt and full
structured target fit the preset training limit. The test set intentionally
includes long records when available so the test-time chunking path can also
be checked.

## 0. Setup

In [1]:
import os
import json
import random

import pandas as pd

PROCESSED_DIR = "data/processed"
INPUT_PATH = os.path.join(PROCESSED_DIR, "cnn_processed.jsonl")
TRAIN_PATH = os.path.join(PROCESSED_DIR, "train.jsonl")
VAL_PATH = os.path.join(PROCESSED_DIR, "validation.jsonl")
TEST_PATH = os.path.join(PROCESSED_DIR, "test.jsonl")

TRAIN_SIZE = 800
VAL_SIZE = 100
TEST_SIZE = 100
PREFERRED_LONG_TEST = 50
SEED = 42

random.seed(SEED)

## 1. Load processed records

In [2]:
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(
        f"Could not find '{INPUT_PATH}'. Run preprocessing_cnn.ipynb first."
    )

records = []
with open(INPUT_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

processed_df = pd.DataFrame(records)
print(f"Loaded {len(processed_df)} records")
print(processed_df["fits_training_limit"].value_counts())

Loaded 2500 records
fits_training_limit
True     1364
False    1136
Name: count, dtype: int64


## 2. Create disjoint train, validation and test sets

In [3]:
fit_df = processed_df[
    processed_df["fits_training_limit"] == True
].sample(frac=1.0, random_state=SEED).reset_index(drop=True)

long_df = processed_df[
    processed_df["fits_training_limit"] == False
].sample(frac=1.0, random_state=SEED).reset_index(drop=True)

required_fit = TRAIN_SIZE + VAL_SIZE
if len(fit_df) < required_fit:
    raise ValueError(
        f"Only {len(fit_df)} complete pairs fit, but {required_fit} are required. "
        "Increase SAMPLE_SIZE in preprocessing_cnn.ipynb."
    )

train_df = fit_df.iloc[:TRAIN_SIZE].copy().reset_index(drop=True)
val_df = fit_df.iloc[TRAIN_SIZE:TRAIN_SIZE + VAL_SIZE].copy().reset_index(drop=True)

used_fit_ids = set(train_df["article_id"]) | set(val_df["article_id"])
remaining_fit_df = fit_df[
    ~fit_df["article_id"].isin(used_fit_ids)
].reset_index(drop=True)

long_test_count = min(PREFERRED_LONG_TEST, len(long_df), TEST_SIZE)
fit_test_count = TEST_SIZE - long_test_count

if len(remaining_fit_df) < fit_test_count:
    extra_long_needed = fit_test_count - len(remaining_fit_df)
    long_test_count = min(TEST_SIZE, long_test_count + extra_long_needed)
    fit_test_count = TEST_SIZE - long_test_count

test_parts = []
if long_test_count:
    test_parts.append(long_df.iloc[:long_test_count])
if fit_test_count:
    test_parts.append(remaining_fit_df.iloc[:fit_test_count])

test_df = pd.concat(test_parts, ignore_index=True)
test_df = test_df.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

if len(test_df) != TEST_SIZE:
    raise ValueError(
        f"Only {len(test_df)} test records could be created. "
        "Increase SAMPLE_SIZE in preprocessing_cnn.ipynb."
    )

# Leakage checks.
assert set(train_df["article_id"]).isdisjoint(val_df["article_id"])
assert set(train_df["article_id"]).isdisjoint(test_df["article_id"])
assert set(val_df["article_id"]).isdisjoint(test_df["article_id"])

print(f"Train:      {len(train_df)} complete pairs")
print(f"Validation: {len(val_df)} complete pairs")
print(
    f"Test:       {len(test_df)} records "
    f"({int((~test_df['fits_training_limit']).sum())} long/non-fitting)"
)

Train:      800 complete pairs
Validation: 100 complete pairs
Test:       100 records (50 long/non-fitting)


## 3. Save splits

In [4]:
def save_jsonl(df: pd.DataFrame, path: str):
    with open(path, "w", encoding="utf-8") as f:
        for record in df.to_dict(orient="records"):
            f.write(json.dumps(record, ensure_ascii=False) + "\n")
    print(f"Saved {len(df)} rows to '{path}'")


save_jsonl(train_df, TRAIN_PATH)
save_jsonl(val_df, VAL_PATH)
save_jsonl(test_df, TEST_PATH)

Saved 800 rows to 'data/processed\train.jsonl'
Saved 100 rows to 'data/processed\validation.jsonl'
Saved 100 rows to 'data/processed\test.jsonl'
